# 3 · Measuring agreement honestly

The engine reproduces a process a human already performs, so there is a ground
truth — their spreadsheet. Comparing against it is where most of the wrong
answers in this project got caught, and it needed three ideas that are not in the
usual precision/recall toolkit.

In [ ]:
import sys
sys.path.insert(0, "..")
import pandas as pd
pd.set_option("display.width", 110)
pd.set_option("display.max_columns", 20)

from src.synthetic import generate
from src.rules import frame_from_names, classify
from src.agreement import agreement, totals, score_rule, base_rate, inverse_control

findings = frame_from_names(generate())
result = classify(findings)
report = agreement(result)
print(report.to_string(index=False))

     category  engine_count  analyst_count  agreed  misrouted  over_detected  missed  precision   recall  net_variance  absolute_error
        PATCH          2288           1807    1752        200            336      55   0.765734 0.969563           481             591
   SUPERSEDED           158            413     130          0             28     283   0.822785 0.314770          -255             311
  NOT_SCANNED           683            453     453        133             97       0   0.663250 1.000000           230             230
  CERTIFICATE           147            106     104         17             26       2   0.707483 0.981132            41              45
   APP_UPDATE            87             98      71          4             12      27   0.816092 0.724490           -11              43
   ENCRYPTION           209            171     169         12             28       2   0.808612 0.988304            38              42
CONTROL_GROUP            49             44      43     

## Idea 1 — disagreement has two causes, and they need opposite fixes

When the engine flags a row the analyst did not flag *in that category*, either:

- **misrouted** — the analyst filed it under a different category. The engine was
  right that it is a finding and wrong about which. That is a *precedence* bug.
- **over-detected** — the analyst did not classify it at all. That is a
  *threshold* bug.

One number blends them and points at neither. Splitting them tells you which part
of the engine to open.

In [ ]:
print(totals(report))

{'engine_count': 3640, 'analyst_count': 3110, 'agreed': 2739, 'misrouted': 371, 'over_detected': 530, 'missed': 371, 'precision': 0.7525, 'recall': 0.8807, 'similarity': 0.6829, 'net_variance': 530, 'absolute_error': 1272}


## Idea 2 — a net variance hides its own magnitude

The first version of this report compared totals per category. Errors of opposite
sign cancel, and a badly broken engine reports a small variance.

In [ ]:
worst = report.assign(
    net=lambda d: d["net_variance"].abs(),
    real=lambda d: d["absolute_error"])[["category", "net", "real"]]
print(worst.to_string(index=False))
print(f"\nsum of net variances (what a totals report shows): "
      f"{report['net_variance'].sum():+,}")
print(f"sum of magnitudes (what is actually wrong):      "
      f"{report['absolute_error'].sum():,}")

     category  net  real
        PATCH  481   591
   SUPERSEDED  255   311
  NOT_SCANNED  230   230
  CERTIFICATE   41    45
   APP_UPDATE   11    43
   ENCRYPTION   38    42
CONTROL_GROUP    5     7
    UNINSTALL    1     3

sum of net variances (what a totals report shows): +530
sum of magnitudes (what is actually wrong):      1,272


The gap between those two numbers is the entire argument for row-level
reconciliation. A totals report is not a weaker version of this — it points the
wrong way.

## Idea 3 — score a rule against its own inverse, not against zero

This is the cheapest way to kill a plausible rule, and it retired three of them.

A rule that fires on a few dozen rows and gets ~10% right sounds weak but real.
Then you score the **exact opposite** rule on the same domain. If it also gets
10%, the axis carries no information — the rule was reading the base rate back.

In [ ]:
# Candidate: within a family of monthly Windows rollups, the older month is
# superseded by the newer one. Reasonable, and the same shape of argument that
# works for browsers.
line = findings["definition_name"].str.extract(r":\s*(.*?)\s*\(", expand=False)
newest_period = findings["period"].groupby(line).transform("max")
domain = findings["shape"].eq("kb_titled")
older_kb = (domain & findings["period"].lt(newest_period)).fillna(False)

control = inverse_control(older_kb, domain, findings, "SUPERSEDED")
for label in ("rule", "inverse"):
    print(f"{label:8} {control[label]}")
print(f"base rate {control['base_rate']}")

rule     {'fired': 50, 'hits': 3, 'other_category': 41, 'unlabelled': 6, 'precision': 0.06, 'recall': 0.0073}
inverse  {'fired': 700, 'hits': 55, 'other_category': 543, 'unlabelled': 102, 'precision': 0.0786, 'recall': 0.1332}
base rate 0.1135


The rule and its inverse land on the same number, and that number is the base
rate. The axis is noise.

Contrast with the version axis, where the rule separates and its inverse does
not:

In [ ]:
from src.rules import is_superseded

comparator = findings["shape"].eq("comparator")
control = inverse_control(is_superseded(findings), comparator, findings, "SUPERSEDED")
for label in ("rule", "inverse"):
    print(f"{label:8} {control[label]}")
print(f"base rate {control['base_rate']}")

rule     {'fired': 158, 'hits': 130, 'other_category': 0, 'unlabelled': 28, 'precision': 0.8228, 'recall': 0.3148}
inverse  {'fired': 2030, 'hits': 165, 'other_category': 1578, 'unlabelled': 287, 'precision': 0.0813, 'recall': 0.3995}
base rate 0.1135


## The ceiling

What is left after the version axis is exhausted is not a rule waiting to be
found. In the generator it is literally uniform noise, because that is what the
real residual measured like: findings the analyst marks superseded for reasons
that are not in the export at all.

Knowing that is worth as much as a rule. It is the difference between *"recall is
0.35 and we don't know why"* and *"recall is 0.35, here is the boundary, and the
rest needs a conversation rather than more code"*.

In [ ]:
residual = result[result["analyst_category"].eq("SUPERSEDED")
                  & ~result["category"].eq("SUPERSEDED")]
print(f"findings the analyst calls superseded and the engine does not: {len(residual):,}")
print()
print(residual.groupby("shape")["finding_id"].size().sort_values(ascending=False).to_string())

findings the analyst calls superseded and the engine does not: 283

shape
comparator    165
kb_titled      58
protocol       38
monthly        10
eol             7
config          5
